# DroneTwin Minimal End-to-End Demo

This notebook demonstrates a complete geometry-based 3D perception workflow for an aerial mapping dataset. It starts from overlapping imagery / COLMAP reconstruction output, cleans the dense point cloud, measures the scene, separates ground from elevated structure, grows surface regions, and assigns coarse semantic classes.

The default path starts from an existing COLMAP dense fusion output because full photogrammetry can take a long time. Set `REBUILD_COLMAP = True` below if you want to rebuild from `data/raw` first.

The notebook calls popup-capable scripts with `--no-view` and renders lightweight sampled results inline under each cell.

## Demo Scope

This notebook is a compact technical record of the pipeline stages and the artifacts each stage produces.

- **Photogrammetry:** raw drone imagery is converted into dense 3D geometry with COLMAP.
- **Point-cloud engineering:** noisy reconstruction output is filtered and downsampled into a usable working cloud.
- **Geometry-based segmentation:** RANSAC, local surface normals, region growing, and rule-based scoring create inspectable labels.
- **Explainability:** every stage writes artifacts and prints counts, dimensions, thresholds, and class summaries so the result can be debugged visually and numerically.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d
import pandas as pd
from IPython.display import display


def find_project_root() -> Path:
    """Find the repo root whether the notebook starts in / or /scripts."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "README.md").exists() and (candidate / "scripts").is_dir():
            return candidate
    raise RuntimeError("Could not find DroneTwin project root")


PROJECT_ROOT = find_project_root()
DENSE_DIR = PROJECT_ROOT / "data" / "processed" / "dense"
PYTHON = sys.executable
MAX_INLINE_POINTS = 80_000


def run_script(label: str, script: str, *args: str) -> subprocess.CompletedProcess[str]:
    """Run one repo script from the project root and print its output."""
    command = [PYTHON, str(PROJECT_ROOT / script), *map(str, args)]
    print(f"\n=== {label} ===")
    print(" ".join(command))
    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout)
    result.check_returncode()
    return result


def require_outputs(*paths: Path) -> None:
    missing = [path for path in paths if not path.exists()]
    if missing:
        missing_text = "\n".join(str(path) for path in missing)
        raise FileNotFoundError(f"Missing expected output(s):\n{missing_text}")
    for path in paths:
        print(f"ok: {path.relative_to(PROJECT_ROOT)}")


def sampled_cloud(path: Path, max_points: int = MAX_INLINE_POINTS):
    """Load a PLY and return a deterministic sample for inline plotting."""
    cloud = o3d.io.read_point_cloud(str(path))
    if cloud.is_empty():
        raise ValueError(f"No points loaded from {path}")
    points = np.asarray(cloud.points)
    colors = np.asarray(cloud.colors) if cloud.has_colors() else None
    if len(points) > max_points:
        rng = np.random.default_rng(7)
        indices = np.sort(rng.choice(len(points), size=max_points, replace=False))
        points = points[indices]
        if colors is not None:
            colors = colors[indices]
    return points, colors


def show_cloud(path: Path, title: str, max_points: int = MAX_INLINE_POINTS) -> None:
    """Show a static sampled 3D cloud inline under the notebook cell."""
    points, colors = sampled_cloud(path, max_points=max_points)
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")
    marker_colors = colors if colors is not None and len(colors) == len(points) else points[:, 2]
    scatter = ax.scatter(
        points[:, 0],
        points[:, 1],
        points[:, 2],
        c=marker_colors,
        s=0.35,
        linewidths=0,
        cmap=None if colors is not None else "viridis",
    )
    if colors is None:
        fig.colorbar(scatter, ax=ax, shrink=0.6, label="Z")
    ax.set_title(f"{title} ({len(points):,} sampled points)")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    extent = np.ptp(points, axis=0)
    ax.set_box_aspect(np.maximum(extent, 1e-9))
    ax.view_init(elev=25, azim=-60)
    plt.tight_layout()
    plt.show()


def show_class_report(path: Path, rows: int = 12) -> None:
    """Display the semantic classification CSV inline."""
    report = pd.read_csv(path)
    if "class_name" in report.columns:
        counts = report["class_name"].value_counts().rename_axis("class_name").reset_index(name="region_count")
        display(counts)
    display(report.head(rows))


print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {PYTHON}")

## Configuration

`REBUILD_COLMAP` is off by default so the demo can use the dense point cloud already generated by `run_colmap.py`. Turn it on only when you want to rerun photogrammetry from the raw image folder.

In [ ]:
REBUILD_COLMAP = False
CLEAN_COLMAP_OUTPUTS = False
VIEW_SEMANTIC_CLASSES = False

RAW_FUSED = DENSE_DIR / "fused-keep-more.ply"
CLEANED = DENSE_DIR / "fused-keep-more-cleaned.ply"
GROUND = DENSE_DIR / "ground.ply"
NON_GROUND = DENSE_DIR / "non_ground.ply"
REGIONS = DENSE_DIR / "non_ground_regions.ply"
REGION_PROXY = DENSE_DIR / "non_ground_regions_proxy.ply"
REGION_LABELS = DENSE_DIR / "non_ground_region_labels.npz"
SEMANTIC = DENSE_DIR / "semantic_classes.ply"
NON_GROUND_SEMANTIC = DENSE_DIR / "non_ground_semantic_classes.ply"
REPORT = DENSE_DIR / "region_class_report.csv"
LEGEND = DENSE_DIR / "semantic_class_legend.csv"

## 1. Photogrammetry: `run_colmap.py`

COLMAP turns overlapping drone images into sparse/dense reconstruction products. The later Open3D scripts use `data/processed/dense/fused-keep-more.ply`.

In [ ]:
if REBUILD_COLMAP:
    colmap_args = ["--clean"] if CLEAN_COLMAP_OUTPUTS else []
    run_script("COLMAP reconstruction", "scripts/run_colmap.py", *colmap_args)
else:
    print("Skipping COLMAP rebuild. Set REBUILD_COLMAP = True to run scripts/run_colmap.py.")

require_outputs(RAW_FUSED)

## 2. Cleanup: `clean_point_cloud.py`

This removes isolated points and downsamples the dense fusion into a cleaner working cloud.

In [ ]:
show_cloud(RAW_FUSED, "Raw fused cloud before cleanup", max_points=60_000)
run_script("Clean dense point cloud", "scripts/clean_point_cloud.py")
require_outputs(CLEANED)
show_cloud(CLEANED, "Cleaned point cloud after filtering/downsampling")

## 3. Analysis: `analyze_point_cloud.py`

This prints basic dimensions, elevation stats, and point-spacing diagnostics. The notebook then renders a sampled cloud inline.

In [ ]:
run_script("Analyze cleaned point cloud", "scripts/analyze_point_cloud.py", "--no-view")
show_cloud(CLEANED, "Cleaned cloud colored by stored color or height")

## 4. Ground Split: `segment_point_cloud.py`

This fits the dominant RANSAC plane as ground and writes separate ground/non-ground PLY files.

In [ ]:
run_script("Segment ground and non-ground", "scripts/segment_point_cloud.py", "--no-view")
require_outputs(GROUND, NON_GROUND)
show_cloud(GROUND, "Ground points")
show_cloud(NON_GROUND, "Non-ground points")

## 5. Region Growing: `region_grow_non_ground.py`

This groups non-ground points into connected surface patches and writes the label sidecar used by semantic classification.

In [ ]:
run_script(
    "Grow non-ground surface regions",
    "scripts/region_grow_non_ground.py",
    "--proxy-output",
    "default",
)
require_outputs(REGIONS, REGION_PROXY, REGION_LABELS)
show_cloud(REGIONS, "Region-grown non-ground patches")

## 6. Semantic Classes: `classify_regions.py`

This converts region features into coarse rule-based classes: ground, tree, water, building, other, and noise.

In [ ]:
classify_args = ["--view"] if VIEW_SEMANTIC_CLASSES else []
run_script("Classify regions", "scripts/classify_regions.py", *classify_args)
require_outputs(SEMANTIC, NON_GROUND_SEMANTIC, REPORT, LEGEND)
show_cloud(SEMANTIC, "Semantic classes")
show_class_report(REPORT)

## Results Summary

This section turns the generated artifacts into reviewable evidence: file sizes, point counts, and semantic class counts.

In [ ]:
def ply_vertex_count(path: Path) -> int | None:
    """Read the PLY header to get point count without loading the full cloud."""
    with path.open("rb") as handle:
        for raw_line in handle:
            line = raw_line.decode("utf-8", errors="ignore").strip()
            if line.startswith("element vertex"):
                return int(line.split()[-1])
            if line == "end_header":
                return None
    return None


artifact_rows = []
for name, path in [
    ("raw fused cloud", RAW_FUSED),
    ("cleaned cloud", CLEANED),
    ("ground cloud", GROUND),
    ("non-ground cloud", NON_GROUND),
    ("region-grown non-ground", REGIONS),
    ("semantic scene", SEMANTIC),
    ("semantic non-ground", NON_GROUND_SEMANTIC),
]:
    artifact_rows.append(
        {
            "artifact": name,
            "path": str(path.relative_to(PROJECT_ROOT)),
            "size_mb": round(path.stat().st_size / (1024 * 1024), 2),
            "points": ply_vertex_count(path),
        }
    )

artifact_summary = pd.DataFrame(artifact_rows)
display(artifact_summary)

report = pd.read_csv(REPORT)
class_summary = (
    report.groupby("class_name", as_index=False)
    .agg(region_count=("region_id", "count"), total_proxy_points=("point_count", "sum"))
    .sort_values("region_count", ascending=False)
)
display(class_summary)

print(
    "Key takeaway: the pipeline reduces a very large COLMAP reconstruction "
    "into inspectable segmented artifacts and explainable class reports."
)

## Technical Takeaways

- The raw COLMAP fusion is large and noisy; the cleanup stage makes it practical to inspect and segment.
- The ground split converts one dense cloud into terrain and elevated structure, which makes later classification easier to reason about.
- Region growing creates object/structure candidates before semantic labeling, so classification is based on surfaces rather than individual noisy points.
- The semantic stage is intentionally rule-based and transparent; the CSV report shows why each region was labeled instead of hiding the decision inside a black-box model.